# 🕵️ Dia 87 — Análise de fraude e engenharia social

**Investigador Cyber 90 Dias** · Fase 7 · Rank Jutsu `S`

> Baseado no *Day 87* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-7-capstone-career/day-87.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).
> **Material sintético.** O *case packet* (Orrin Valley / Castellan) e a Operação Isca são ficção; domínios `.example`, IPs de documentação.

---

### 🎬 Onde paramos

*"Não ligue para a agência: as linhas estão congestionadas."* Rafa perguntou por que essa frase está na mensagem do falso atendimento. Resposta: porque uma ligação teria acabado com o golpe. Toda frase de golpe bem feito é uma contramedida.

Neste notebook você vai:

1. **Pontuar mecanismos com citação obrigatória**: "presente" só vale se a frase estiver no texto.
2. **Extrair as inconsistências da carta bancária** automaticamente.
3. **Gerar o diagrama de raias em Mermaid** com os pontos de quebra.
4. **Testar o cartão de padrão** com um detector de vazamento de identificadores.
5. **Fazer tudo de novo com o SMS, a página e o WhatsApp da Operação Isca.**

🐍 Só biblioteca padrão do Python. Funciona offline: se a rede falhar, o notebook usa a cópia embutida dos arquivos. Rode as células de cima para baixo (Shift+Enter).

## 📦 O pacote do caso (plano A: GitHub; plano B: cópia embutida)

A próxima célula guarda uma **cópia embutida** dos arquivos do *case packet* que este dia usa. Ela é o plano B: se você estiver sem internet, ou se o GitHub estiver fora, o notebook continua funcionando.

A célula seguinte tenta o plano A: baixar cada arquivo do repositório original com `urllib`. Se baixar, compara com a cópia embutida pelo SHA-256 e avisa se o original mudou desde que este notebook foi gerado (é exatamente o raciocínio de custódia do Dia 83).

In [ ]:
# Cópia embutida (plano B). Cada valor é o texto exato do arquivo do case packet.
# Não edite: se mudar um caractere, o SHA-256 muda e a custódia quebra.
EMBUTIDO = {
"01-engagement-letter.md": "# Engagement letter and intake notes\n\n> SYNTHETIC TRAINING MATERIAL. Orrin Valley Millworks, Castellan Hardwood Supply, Marrowline Freight Services and every person named here are fictional. Any resemblance to a real organization or person is coincidental.\n\n## Part A: engagement letter (from the client)\n\nFrom: Owen Tran, Operations Director, Orrin Valley Millworks\nTo: You, contract investigator\nDate: Tuesday 17 March 2026\nRe: Diverted vendor payment, request for investigation\n\nOwen writes:\n\nWe are a 64-person custom millwork and cabinetry manufacturer. On Friday 13 March our weekly payment run sent $48,612.50 for invoice CHS-20417 to a bank account that our long-standing lumber supplier, Castellan Hardwood Supply, says is not theirs. We found out on Monday 16 March at about 10:15 our time, when Castellan's accounts receivable manager, Dana Whitlock, phoned to ask why the invoice was unpaid.\n\nWhat we have done so far:\n\n1. Our bank has opened a recall request on the payment. We have filed a police report (reference OV-2026-03-0412) and notified our cyber insurer.\n2. Our IT contractor reset Jordan Pike's password, signed out all of Jordan's sessions, and deleted an inbox rule they found on Jordan's mailbox. They did this on 16 March before exporting any logs. They have since exported what we are sending you.\n3. We have not contacted anyone else about this.\n\nWe would like you to answer these questions:\n\n1. How did this happen, step by step?\n2. Was Jordan's email account taken over? If so, when, from where, and what did the attacker do with it?\n3. The fraudulent email quoted Castellan's real invoice, word for word. Did that information leak from us or from Castellan?\n4. Is the same group targeting other suppliers or other staff here? Is there anything else we should block?\n5. What should we change so this does not happen again?\n\nOur insurer has asked for a written report. Please treat anything you tell us as something the insurer and possibly the police will read.\n\n## Part B: authorization and scope\n\nOrrin Valley Millworks authorizes you to:\n\n- Review every file in this case packet, including the three log exports under `logs/`.\n- Run passive research on the domains, IP addresses, email addresses and phone numbers that appear in the packet, using the pre-collected lookup results in `06-lookup-results.md`. Because every domain and IP here is a reserved documentation value, live lookups return nothing useful. The lookup file stands in for what WHOIS, passive DNS, certificate transparency and a URL scanner returned when the client's IT contractor ran them on 17 March.\n- Treat the two staff statements in Part C as your interview evidence. You cannot interview the staff again before the report is due.\n- Produce a case graph and a written report for the client, its insurer and, if the client chooses, the police.\n\nThe following are out of scope. The client did not authorize them, and several would be unlawful or unsafe in a real case:\n\n- Visiting, logging in to, or submitting anything to `cstl-docshare.example` or any other attacker-controlled site, even \"just to see the page.\"\n- Contacting the attacker by email (including the freemail Reply-To address), by phone (555-0148) or any other channel, or replying to the fraudulent thread.\n- Investigating Castellan Hardwood Supply's systems, mailboxes or staff. Castellan is a separate organization and has not engaged you. You may record what Castellan told the client and recommend that the client share indicators with Castellan.\n- Researching Dana Whitlock, Jordan Pike, Maren Beaulieu or any other named person's personal social media, home address, relatives or other personal life. You are investigating what happened to accounts and money, not the private lives of the people involved.\n- Researching Marrowline Freight Services, the real business whose brand appears on the attacker's infrastructure, beyond noting that its brand was used.\n- Attempting to identify the attacker as a named individual. Attribution to a person is a police matter. You may describe infrastructure, tactics and patterns.\n- Tracing, freezing or recovering the funds. The bank and police are handling that.\n- Any active scanning of any IP address or domain.\n\n## Part C: intake notes (taken by you on the intake call, 17 March 2026)\n\nEnvironment facts from the client's IT contractor (Priya Anand, contractor):\n\n- Email is a cloud-hosted mailbox service with a webmail interface at `mail.orrinvalley.example`. All sign-ins go through the same cloud identity service.\n- MFA rollout is 70% complete. Jordan Pike's account (`jpike`) was not yet enrolled. The controller, Maren Beaulieu (`mbeaulieu`), was enrolled in February.\n- The inbound mail gateway tags outside mail with `[EXTERNAL]` in the subject. It does not rewrite links and does not flag lookalike domains.\n- The web proxy logs in the appliance's local time. Priya is \"pretty sure\" it is set to the office time zone (US/Canada Eastern, IANA zone `America/Toronto`) but has not checked.\n- The mailbox audit export covers all events for `jpike` and `mbeaulieu` from 9 March 2026 00:00 UTC to 17 March 2026 12:00 UTC, plus failed sign-ins tenant-wide from one IP that Priya noticed. It does not cover the shared `ap@` mailbox or any other staff. Audit data before 9 March was not exported.\n- The mail gateway trace covers 1 to 17 March, filtered to messages involving `castellan`, `jpike` or `mbeaulieu`.\n- Vendor master changes are made in the ERP by the controller. ERP logs were not provided. According to Maren, she changed Castellan's bank account and remittance email address on 11 March after an email from Jordan.\n\nStatement from Jordan Pike, AP specialist (summarized):\n\n> \"I got the email from Dana on Tuesday morning, the 10th. It was in the same thread as the invoice, so I didn't think twice. I clicked the link, it asked me to sign in to see the document, and I did. Then it just went to Castellan's normal website. I figured the document had moved or something. I never sent Maren anything about banking. I didn't see any more emails from Castellan that week, which I thought was a bit odd because Dana usually sends a reminder before the due date.\"\n\nStatement from Maren Beaulieu, controller (summarized):\n\n> \"On Wednesday the 11th Jordan emailed me Castellan's bank letter and asked me to update the vendor record before Friday's run. The email said Castellan's phones were down that week, so I didn't call. The letter looked right. I made the change and replied to Jordan.\"\n\nWhat Castellan told Owen by phone on 16 March (hearsay, one call, not verified):\n\n> Dana Whitlock said she never sent a banking-change email. Castellan's IT \"checked and found nothing.\" Dana also mentioned that \"a couple of other customers\" had asked about a banking change in the past two weeks.\n",
"02-suspicious-email.eml": "X-Case-Packet-Notice: SYNTHETIC TRAINING DATA. Fictional organizations, people, domains (RFC 2606 .example) and IPs (RFC 5737). Do not look up or contact anything in this file.\nReturn-Path: <bounce-7731@castellan-hardwood.example>\nReceived: from gw01.orrinvalley.example (gw01.orrinvalley.example [192.0.2.25])\n\tby mbx01.orrinvalley.example with ESMTPS id 4f1c9a2e\n\tfor <jpike@orrinvalley.example>; Tue, 10 Mar 2026 14:02:39 +0000\nReceived: from mail.castellan-hardwood.example (vps-47.pinecrate.example [203.0.113.47])\n\tby gw01.orrinvalley.example with ESMTPS id 8b3e0d17\n\tfor <jpike@orrinvalley.example>; Tue, 10 Mar 2026 14:02:37 +0000\nReceived: from [10.8.0.6] (unknown [198.51.100.23])\n\t(Authenticated sender: ar@castellan-hardwood.example)\n\tby mail.castellan-hardwood.example (Postfix) with ESMTPSA id 2C4A1E03\n\tfor <jpike@orrinvalley.example>; Tue, 10 Mar 2026 14:02:35 +0000\nAuthentication-Results: gw01.orrinvalley.example;\n\tspf=pass (gw01.orrinvalley.example: domain of bounce-7731@castellan-hardwood.example designates 203.0.113.47 as permitted sender) smtp.mailfrom=castellan-hardwood.example;\n\tdkim=none (message not signed);\n\tdmarc=none action=none header.from=castellan-hardwood.example\nX-OVM-Gateway-Verdict: clean; score=3.1; threshold=5.0; tags=EXTERNAL\nDate: Tue, 10 Mar 2026 10:02:31 -0400\nFrom: \"Dana Whitlock | Castellan Hardwood Supply\" <dana.whitlock@castellan-hardwood.example>\nReply-To: <dwhitlock.castellan@freemail.example>\nTo: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] RE: Invoice CHS-20417 - updated remittance details\nMessage-ID: <20260310140231.2C4A1E03@mail.castellan-hardwood.example>\nIn-Reply-To: <20260303144702.GA11873@mail.castellanhardwood.example>\nReferences: <20260303144702.GA11873@mail.castellanhardwood.example>\nUser-Agent: Mozilla/5.0 (X11; Linux x86_64; rv:115.0) Gecko/20100101 Thunderbird/115.8.0\nMIME-Version: 1.0\nContent-Type: multipart/alternative; boundary=\"------------b7Qx2LmR0v8aZ3\"\n\nThis is a multi-part message in MIME format.\n--------------b7Qx2LmR0v8aZ3\nContent-Type: text/plain; charset=UTF-8\nContent-Transfer-Encoding: 7bit\n\nHi Jordan,\n\nFollowing up on the invoice below. As part of our annual audit we have\nmoved our receivables to a new account, effective this week. Please use\nthe updated remittance details for CHS-20417 and for all future payments.\nThe signed bank letter and the vendor update form are on our secure\nportal:\n\nView remittance form (secure): https://castellanhardwood.example/remittance\n\nOur office phones are being migrated to a new system this week, so email\nis the quickest way to reach me. We would appreciate it if CHS-20417 could\ngo out in Friday's payment run, as the old account closes on the 16th.\n\nThanks so much,\n\nDana Whitlock\nAccounts Receivable Manager\nCastellan Hardwood Supply\nTel. 555-0148 | ar@castellan-hardwood.example\n\nOn Tue, Mar 3, 2026 at 9:47 AM Dana Whitlock <dwhitlock@castellanhardwood.example> wrote:\n> Good morning,\n>\n> Please find attached invoice CHS-20417 for the February white oak and\n> hard maple order (PO 7719), total $48,612.50, net 10.\n>\n> Thank you for your business.\n>\n> Dana Whitlock\n> Accounts Receivable Manager\n> Castellan Hardwood Supply\n> Tel. 555-0112 | accounts@castellanhardwood.example\n\n--------------b7Qx2LmR0v8aZ3\nContent-Type: text/html; charset=UTF-8\nContent-Transfer-Encoding: 7bit\n\n<html><body style=\"font-family:Calibri,Arial,sans-serif;font-size:11pt\">\n<p>Hi Jordan,</p>\n<p>Following up on the invoice below. As part of our annual audit we have moved our receivables to a new account, effective this week. Please use the updated remittance details for CHS-20417 and for all future payments. The signed bank letter and the vendor update form are on our secure portal:</p>\n<p><b>View remittance form (secure):</b> <a href=\"https://cstl-docshare.example/view/r?u=b3JyaW52YWxsZXkuZXhhbXBsZQ%3D%3D&amp;t=anBpa2U%3D\">https://castellanhardwood.example/remittance</a></p>\n<p>Our office phones are being migrated to a new system this week, so email is the quickest way to reach me. We would appreciate it if CHS-20417 could go out in Friday's payment run, as the old account closes on the 16th.</p>\n<p>Thanks so much,</p>\n<p>Dana Whitlock<br>Accounts Receivable Manager<br>Castellan Hardwood Supply<br>Tel. 555-0148 | ar@castellan-hardwood.example</p>\n<img src=\"https://castellanhardwood.example/img/logo.png\" width=\"160\" alt=\"Castellan Hardwood Supply\">\n<div style=\"border-left:2px solid #ccc;padding-left:8px;color:#555\">\n<p>On Tue, Mar 3, 2026 at 9:47 AM Dana Whitlock &lt;dwhitlock@castellanhardwood.example&gt; wrote:</p>\n<p>Good morning,</p>\n<p>Please find attached invoice CHS-20417 for the February white oak and hard maple order (PO 7719), total $48,612.50, net 10.</p>\n<p>Thank you for your business.</p>\n<p>Dana Whitlock<br>Accounts Receivable Manager<br>Castellan Hardwood Supply<br>Tel. 555-0112 | accounts@castellanhardwood.example</p>\n</div>\n</body></html>\n\n--------------b7Qx2LmR0v8aZ3--\n",
"03-related-messages.md": "# Related messages\n\n> SYNTHETIC TRAINING MATERIAL. All organizations, people, domains, IPs and account numbers are fictional.\n\nThe client's IT contractor pulled these messages from mailboxes on 16 and 17 March 2026. Headers are trimmed to the fields that matter; the full suspicious email is in `02-suspicious-email.eml`. Message IDs (M1 to M7) are yours to use in notes and on the case graph.\n\n## M1: the real invoice (3 March)\n\nWhere found: shared mailbox `ap@orrinvalley.example`, Inbox.\n\n```\nDate: Tue, 3 Mar 2026 09:47:02 -0500\nFrom: Dana Whitlock <dwhitlock@castellanhardwood.example>\nTo: ap@orrinvalley.example\nCc: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] Invoice CHS-20417\nMessage-ID: <20260303144702.GA11873@mail.castellanhardwood.example>\nAuthentication-Results: gw01.orrinvalley.example; spf=pass smtp.mailfrom=castellanhardwood.example;\n\tdkim=pass header.d=castellanhardwood.example; dmarc=pass (p=quarantine) header.from=castellanhardwood.example\nAttachment: CHS-20417.pdf\n```\n\n```\nGood morning,\n\nPlease find attached invoice CHS-20417 for the February white oak and\nhard maple order (PO 7719), total $48,612.50, net 10.\n\nThank you for your business.\n\nDana Whitlock\nAccounts Receivable Manager\nCastellan Hardwood Supply\nTel. 555-0112 | accounts@castellanhardwood.example\n```\n\nThe attached invoice PDF names the payee as \"Castellan Hardwood Supply Inc.\" with remittance to Vellacourt Commercial Bank, account ending 4471. It has been Castellan's account on file since 2019.\n\n## M2: the suspicious email (10 March)\n\nSee `02-suspicious-email.eml`.\n\n## M3: \"Jordan\" to the controller (11 March)\n\nWhere found: `mbeaulieu` Inbox. No copy exists in Jordan's Sent Items or Deleted Items.\n\n```\nDate: Wed, 11 Mar 2026 09:52:44 -0400\nFrom: Jordan Pike <jpike@orrinvalley.example>\nTo: Maren Beaulieu <mbeaulieu@orrinvalley.example>\nSubject: Castellan banking update - vendor change for Friday run\nMessage-ID: <a81f3c07e2d94b6e@mbx01.orrinvalley.example>\nAttachment: Castellan_Bank_Letter_2026.pdf\n```\n\n```\nHi Maren,\n\nCastellan sent over their updated banking letter (attached) - they moved\nbanks as part of their annual audit and the old account closes on the 16th.\nCan you update the vendor master before Friday's run? They also asked that\nremittance advices go to ar@castellan-hardwood.example from now on.\n\nDana mentioned their phones are down this week while they switch systems,\nso no need to call, she said email is best.\n\nThanks!\nJordan\n```\n\nText of the attachment, `Castellan_Bank_Letter_2026.pdf`, as extracted by IT:\n\n```\nCASTELLAN HARDWOOD SUPPLY LTD.\n1400 Ferrule Road, Wrenmoor Crossing  |  Tel. 555-0148  |  ar@castellan-hardwood.example\n\n9 March 2026\n\nTo our valued customers,\n\nRE: CHANGE OF BANKING DETAILS\n\nFollowing our annual financial audit, Castellan Hardwood Supply has\ntransferred its receivables to a new banking partner. Effective immediately,\nplease remit all payments to:\n\n    Bank:            Pellbrook Savings & Trust\n    Account name:    Castellan Hardwood Supply Ltd.\n    Account ending:  8830\n\nOur previous account will be closed on 16 March 2026. Payments sent to the\nprevious account after that date will be returned and may incur late fees.\n\nPlease direct any questions to our Accounts Receivable team at\nar@castellan-hardwood.example.\n\nSincerely,\n\nDana Whitlock                          G. Castellan\nAccounts Receivable Manager            Chief Financial Officer\n```\n\nDocument metadata for the attachment, from `exiftool Castellan_Bank_Letter_2026.pdf` run by IT:\n\n```\nFile Size                       : 41 kB\nPDF Version                     : 1.7\nPage Count                      : 1\nProducer                        : LibreOffice 7.6\nCreator                         : Writer\nAuthor                          : user\nTitle                           : vendor_letter_v3\nCreate Date                     : 2026:02:26 19:12:44-05:00\nModify Date                     : 2026:03:09 22:41:03-04:00\n```\n\nSHA-256 (as recorded by the mail gateway): `5d0c7a1e9b43f26c8e7a0d51b9f4c3e2a6d8b17f04e9c5a2d3b6f8e1c7a9042b`\n\n## M4: the controller's reply (11 March)\n\nWhere found: `mbeaulieu` Sent Items, and Jordan's `RSS Subscriptions` folder, marked as read.\n\n```\nDate: Wed, 11 Mar 2026 11:06:12 -0400\nFrom: Maren Beaulieu <mbeaulieu@orrinvalley.example>\nTo: Jordan Pike <jpike@orrinvalley.example>\nSubject: RE: Castellan banking update - vendor change for Friday run\n```\n\n```\nDone, updated in the vendor master (bank + remittance email). Please save\nthe letter to the Castellan vendor file. M.\n```\n\n## M5: the real reminder (12 March)\n\nWhere found: Jordan's `RSS Subscriptions` folder, marked as read. The copy in `ap@` Inbox is unread.\n\n```\nDate: Thu, 12 Mar 2026 12:05:38 -0400\nFrom: Dana Whitlock <dwhitlock@castellanhardwood.example>\nTo: ap@orrinvalley.example\nCc: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] Reminder: CHS-20417 due 13 March\nAuthentication-Results: gw01.orrinvalley.example; spf=pass smtp.mailfrom=castellanhardwood.example;\n\tdkim=pass header.d=castellanhardwood.example; dmarc=pass (p=quarantine) header.from=castellanhardwood.example\n```\n\n```\nHi Jordan,\n\nFriendly reminder that CHS-20417 ($48,612.50) is due tomorrow. Our\nbanking details are unchanged from the invoice.\n\nThanks,\nDana\nTel. 555-0112\n```\n\n## M6: the attacker's follow-up (12 March)\n\nWhere found: Jordan's `RSS Subscriptions` folder, marked as read.\n\n```\nDate: Thu, 12 Mar 2026 12:39:58 -0400\nFrom: \"Dana Whitlock | Castellan Hardwood Supply\" <dana.whitlock@castellan-hardwood.example>\nReply-To: <dwhitlock.castellan@freemail.example>\nTo: Jordan Pike <jpike@orrinvalley.example>\nSubject: [EXTERNAL] RE: Invoice CHS-20417 - updated remittance details\nAuthentication-Results: gw01.orrinvalley.example; spf=pass smtp.mailfrom=castellan-hardwood.example;\n\tdkim=none; dmarc=none header.from=castellan-hardwood.example\n```\n\n```\nHi Jordan,\n\nJust confirming the update went through on your side. Could you send the\nremittance advice as soon as the payment is released tomorrow? Our auditors\nneed it for the account transition.\n\nThanks again for your help with this,\nDana\n```\n\n## M7: the automated remittance advice (13 March)\n\nWhere found: mail gateway trace only (outbound). The ERP sends remittance advices automatically to the remittance address on the vendor record.\n\n```\nDate: Fri, 13 Mar 2026 15:02:07 -0400\nFrom: Orrin Valley Millworks Payables <erp-noreply@orrinvalley.example>\nTo: ar@castellan-hardwood.example\nSubject: Remittance advice - Orrin Valley Millworks - payment 000731\n```\n\n```\nPayment 000731 has been released.\nPayee: Castellan Hardwood Supply Ltd.   Amount: $48,612.50\nInvoices: CHS-20417\nRemitted to: Pellbrook Savings & Trust, account ending 8830\n```\n",
"04-vendor-site-archived.md": "# Artifact A: Castellan Hardwood Supply \"About us\" page (archived)\n\n> SYNTHETIC TRAINING MATERIAL. Castellan Hardwood Supply, Wrenmoor Crossing and every person named here are fictional.\n\nSource: web archive capture of `https://castellanhardwood.example/about`, captured 2 November 2025 at 16:20 UTC. The client's IT contractor saved the rendered text and the page head on 17 March 2026. This is the genuine vendor's site as it looked before the incident.\n\n## Page head (excerpt)\n\n```html\n<title>About Us | Castellan Hardwood Supply</title>\n<meta name=\"description\" content=\"Family-owned hardwood lumber yard supplying cabinetmakers and furniture builders since 1986.\">\n<link rel=\"icon\" href=\"/favicon.ico\">\n<link rel=\"stylesheet\" href=\"/css/site.css?v=2025.3\">\n```\n\n## Rendered text\n\n> **About Castellan Hardwood Supply**\n>\n> Castellan Hardwood Supply has milled and supplied North American hardwoods to cabinetmakers, millwork shops and furniture builders since 1986. We are a family-owned business, now in our third generation, operating from our yard and kiln facility at 1400 Ferrule Road in Wrenmoor Crossing.\n>\n> We stock white oak, red oak, hard and soft maple, cherry, walnut and ash in 4/4 through 12/4, with custom milling to your cut list. Most orders within our delivery region ship within five business days.\n>\n> **Our team**\n>\n> Graham Castellan, President\n> Ines Castellan-Rowe, Chief Financial Officer\n> Tobias Venn, Yard and Kiln Manager\n> Dana Whitlock, Accounts Receivable Manager\n>\n> **Contact**\n>\n> Orders: orders@castellanhardwood.example\n> Accounts: accounts@castellanhardwood.example\n> Phone: 555-0112 (Monday to Friday, 7:00 to 16:00)\n>\n> **Payments**\n>\n> For your security, Castellan Hardwood Supply will never change its banking details by email alone. If you receive a request to change how you pay us, call our office at the number above before acting on it.\n>\n> © 2025 Castellan Hardwood Supply Inc.\n",
"06-lookup-results.md": "# Pre-collected lookup results\n\n> SYNTHETIC TRAINING MATERIAL. Every domain uses the reserved `.example` name, every IP is an RFC 5737 documentation address, and every AS number is an RFC 5398 documentation ASN. Registrars, hosting companies and carriers named here are fictional.\n\nThe client's IT contractor ran these passive lookups on 17 March 2026 between 12:40 and 13:30 UTC, using WHOIS, a passive DNS service, a certificate transparency search, a URL scanning service and a favicon-hash search. In a real case you would run them yourself and log each one. Here, reading a section counts as running that lookup: log it in your collection log with the section number as the source and 17 March 2026 as the access date.\n\nSection numbers (L1 to L9) are for citation in your notes and case graph.\n\n## L1: WHOIS\n\n| Field | castellanhardwood.example | castellan-hardwood.example | cstl-docshare.example | marrowline-docs.example | quennmarsh-accounts.example |\n|---|---|---|---|---|---|\n| Created | 2004-06-14 | 2026-02-24T21:14:09Z | 2026-02-26T03:40:51Z | 2026-02-19T22:05:33Z | 2026-03-14T02:51:17Z |\n| Registrar | Beaconhill Domains | Quillfeather Names | Quillfeather Names | Quillfeather Names | Quillfeather Names |\n| Registrant org | Castellan Hardwood Supply Inc. | REDACTED FOR PRIVACY | REDACTED FOR PRIVACY | REDACTED FOR PRIVACY | REDACTED FOR PRIVACY |\n| Name servers | ns1/ns2.cedarlinedns.example | ns1/ns2.hollowdns.example | ns1/ns2.hollowdns.example | ns1/ns2.hollowdns.example | ns1/ns2.hollowdns.example |\n| Expires | 2027-06-14 | 2027-02-24 | 2027-02-26 | 2027-02-19 | 2027-03-14 |\n\nContext from the registrar and DNS host's own public pages: Quillfeather Names is a low-cost registrar with about 3 million domains under management. Hollowdns is a free DNS hosting service with about 90,000 zones.\n\n## L2: current DNS records (17 March 2026)\n\n```\ncastellanhardwood.example.          A      192.0.2.140\ncastellanhardwood.example.          MX     10 mail.castellanhardwood.example.\nmail.castellanhardwood.example.     A      198.51.100.180\ncastellanhardwood.example.          TXT    \"v=spf1 ip4:198.51.100.180 -all\"\n_dmarc.castellanhardwood.example.   TXT    \"v=DMARC1; p=quarantine; rua=mailto:dmarc@castellanhardwood.example\"\n\ncastellan-hardwood.example.         A      203.0.113.47\ncastellan-hardwood.example.         MX     10 mail.castellan-hardwood.example.\nmail.castellan-hardwood.example.    A      203.0.113.47\ncastellan-hardwood.example.         TXT    \"v=spf1 ip4:203.0.113.47 -all\"\n_dmarc.castellan-hardwood.example.  NXDOMAIN\n\ncstl-docshare.example.              A      203.0.113.88\nwww.cstl-docshare.example.          CNAME  cstl-docshare.example.\n\nmarrowline-docs.example.            A      203.0.113.91\nportal.marrowline-docs.example.     A      203.0.113.91\nmail.marrowline-docs.example.       NXDOMAIN\n\nquennmarsh-accounts.example.        A      203.0.113.91\n```\n\n## L3: passive DNS history\n\n| rrname | Type | rdata | First seen (UTC) | Last seen (UTC) |\n|---|---|---|---|---|\n| castellanhardwood.example | A | 192.0.2.140 | 2019-04-02 | 2026-03-17 |\n| mail.castellanhardwood.example | A | 198.51.100.170 | 2017-08-11 | 2026-01-19 |\n| mail.castellanhardwood.example | A | 198.51.100.180 | 2026-01-19 | 2026-03-17 |\n| castellan-hardwood.example | A | 203.0.113.47 | 2026-02-25 | 2026-03-17 |\n| mail.castellan-hardwood.example | A | 203.0.113.47 | 2026-02-25 | 2026-03-17 |\n| cstl-docshare.example | A | 203.0.113.88 | 2026-02-27 | 2026-03-17 |\n| marrowline-docs.example | A | 203.0.113.91 | 2026-02-20 | 2026-03-17 |\n| portal.marrowline-docs.example | A | 203.0.113.91 | 2026-02-20 | 2026-03-17 |\n| mail.marrowline-docs.example | A | 198.51.100.23 | 2026-02-20 | 2026-03-09 |\n| quennmarsh-accounts.example | A | 203.0.113.91 | 2026-03-15 | 2026-03-17 |\n\nReverse lookups on the same service:\n\n| IP | Hostnames seen resolving to it | PTR record |\n|---|---|---|\n| 198.51.100.23 | mail.marrowline-docs.example (2026-02-20 to 2026-03-09) | vps-23.pinecrate.example |\n| 203.0.113.47 | castellan-hardwood.example, mail.castellan-hardwood.example | vps-47.pinecrate.example |\n| 203.0.113.91 | marrowline-docs.example, portal.marrowline-docs.example, quennmarsh-accounts.example | none |\n\n## L4: certificate transparency\n\n| Log entry ID | Names on certificate | Issuer | Not before (UTC) |\n|---|---|---|---|\n| 9048820415 | marrowline-docs.example, portal.marrowline-docs.example | Let's Encrypt | 2026-02-20 01:37 |\n| 9048820911 | mail.marrowline-docs.example | Let's Encrypt | 2026-02-20 01:39 |\n| 9051182231 | castellan-hardwood.example, www.castellan-hardwood.example, mail.castellan-hardwood.example | Let's Encrypt | 2026-02-25 06:02 |\n| 9051497710 | cstl-docshare.example, www.cstl-docshare.example | Let's Encrypt | 2026-02-27 04:11 |\n| 9063310054 | quennmarsh-accounts.example | Let's Encrypt | 2026-03-15 00:22 |\n| (routine) | castellanhardwood.example, www.castellanhardwood.example | Let's Encrypt | Renewed every 60 to 90 days since 2019; latest 2025-12-30 |\n\n## L5: co-hosting (reverse IP)\n\n| IP | Hosting | Domains currently hosted | Sample |\n|---|---|---|---|\n| 203.0.113.47 | Pinecrate VPS, single-tenant virtual server | 2 hostnames | See L3 |\n| 203.0.113.88 | Pinecrest Shared Hosting, shared web server | 1,412 domains | ferncastle-bakery.example, oakhollow-dental.example, rivermint-yoga.example, tidepost-news.example, cstl-docshare.example, (1,407 more) |\n| 203.0.113.91 | Driftwood Cloud VPS, single-tenant virtual server | 3 hostnames | See L3 |\n\n## L6: URL scan results\n\nS1. Scanned by the client's IT contractor, 17 March 2026 13:20 UTC, sandboxed scanner.\n\n```\nURL:         https://cstl-docshare.example/view/r?u=b3JyaW52YWxsZXkuZXhhbXBsZQ%3D%3D&t=anBpa2U%3D\nFinal URL:   (same)            Status: 200        IP: 203.0.113.88\nPage title:  Secure Document Viewer\nPage text:   \"Castellan Hardwood Supply shared a secure document with you.\n              Sign in with your work email to view Remittance_Form.pdf\"\nResources:   /assets/js/vw-portal.min.js   sha256 9e41c0b7d2a85f3e16c47d09ab3f58e2c1d06b7a94e2f35c8b01d7e6a4c9f218\n             /assets/css/vw.css\n             /favicon.ico                  mmh3 -1742093313\n             https://castellanhardwood.example/img/logo.png   (third-party image load)\nForm:        POST /view/auth   fields: email (prefilled \"jpike@orrinvalley.example\"), password\nNotes:       https://cstl-docshare.example/ returns 404. Scanner followed no redirect.\n```\n\nS2. From a public scan database. Submitted by an unknown third party on 22 February 2026 09:14 UTC.\n\n```\nURL:         https://portal.marrowline-docs.example/view/r?u=[redacted by database]\nStatus:      200        IP: 203.0.113.91\nPage title:  Secure Document Viewer\nPage text:   \"Marrowline Freight Services shared a secure document with you.\n              Sign in with your work email to view Updated_Payment_Instructions.pdf\"\nResources:   /assets/js/vw-portal.min.js   sha256 9e41c0b7d2a85f3e16c47d09ab3f58e2c1d06b7a94e2f35c8b01d7e6a4c9f218\n             /assets/css/vw.css\n             /favicon.ico                  mmh3 -1742093313\n             https://marrowlinefreight.example/images/brand.png   (third-party image load)\nForm:        POST /view/auth   fields: email, password\n```\n\nS3. From a public scan database. Submitted by an unknown third party on 8 October 2025 16:02 UTC.\n\n```\nURL:         https://files.brightquay-legal.example/\nStatus:      200        IP: 192.0.2.201\nPage title:  Secure Document Viewer | Brightquay Legal\nResources:   /static/viewer.js             sha256 3b7f0e92c4d15a8e6f20b9c7d41e5a3f8c62d09e1b4a7f35e8d0c6b2a91f4e07\n             /favicon.ico                  mmh3 -1742093313\nForm:        POST /sso/login\nPage footer: \"Built on the OpenDocViewer template\"\nCert history: certificates for files.brightquay-legal.example continuously since 2021\n```\n\n## L7: favicon hash search\n\n| mmh3 hash | Hosts returned |\n|---|---|\n| -1742093313 | cstl-docshare.example, portal.marrowline-docs.example, files.brightquay-legal.example |\n| 884215106 | castellanhardwood.example, castellan-hardwood.example |\n\n## L8: IP enrichment\n\n| IP | ASN | Network owner | Network type | Location (as reported by the geolocation service) | Other context |\n|---|---|---|---|---|---|\n| 192.0.2.10 | AS64496 | Wrenmoor Fibre | Business fibre | Client's metro area | Orrin Valley office NAT address (confirmed by IT) |\n| 192.0.2.25 | AS64496 | Wrenmoor Fibre | Business fibre | Client's metro area | Orrin Valley mail gateway gw01 (confirmed by IT) |\n| 192.0.2.140 | AS64500 | Cedarline Web Hosting | Managed hosting | Client's region | Castellan's real website |\n| 192.0.2.201 | AS64500 | Cedarline Web Hosting | Managed hosting | Client's region | Brightquay Legal document portal |\n| 198.51.100.23 | AS64510 | Pinecrate VPS | Datacenter / VPS | EU-West datacenter | None |\n| 198.51.100.77 | AS64508 | Skerrow Mobile | Mobile carrier, carrier-grade NAT | Client's metro area | Shared by many mobile subscribers |\n| 198.51.100.170 | AS64502 | Northfold Mail Services | Email hosting | Client's region | Castellan's previous mail server |\n| 198.51.100.180 | AS64502 | Northfold Mail Services | Email hosting | Client's region | Castellan's current mail server |\n| 203.0.113.47 | AS64510 | Pinecrate VPS | Datacenter / VPS | EU-West datacenter | None |\n| 203.0.113.88 | AS64505 | Pinecrest Shared Hosting | Shared web hosting | North America | 1,412 co-hosted domains (L5) |\n| 203.0.113.91 | AS64511 | Driftwood Cloud | Datacenter / VPS | North America | None |\n| 203.0.113.200 | AS64507 | Keelson Hosting | Datacenter | Asia-Pacific datacenter | Listed on a public blocklist for password spraying against cloud mail tenants since January 2026 |\n\nGeolocation for datacenter and VPS addresses reports where the server sits, which says nothing about where its operator is.\n\n## L9: other lookups\n\n| Item | Result |\n|---|---|\n| Phone 555-0112 | Landline. Carrier Wrenmoor Telephone. Directory listing: Castellan Hardwood Supply Inc., listed since 1998. |\n| Phone 555-0148 | VoIP. Carrier Callwisp VoIP. No directory listing. |\n| Domain `freemail.example` | Large consumer webmail provider. Anyone can register an address. No lookup was run on the address `dwhitlock.castellan@freemail.example` itself (out of scope). |\n| Public phishing feed | `marrowline-docs.example` reported 24 February 2026. `cstl-docshare.example` reported 12 March 2026 14:30 UTC by a reporter outside the client. `castellan-hardwood.example` and `quennmarsh-accounts.example` not listed. |\n"
}

In [ ]:
import hashlib                     # SHA-256: a impressão digital de cada arquivo
import urllib.request              # baixar sem instalar nada (requests não é stdlib)

RAW = "https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-7-capstone-career/case-packet/"   # endereço "cru" dos arquivos no GitHub

def baixar(nome, timeout=10):
    """Plano A: baixa o arquivo do GitHub. Devolve o texto ou None se falhar."""
    try:
        with urllib.request.urlopen(RAW + nome, timeout=timeout) as resp:   # timeout: não travar o Colab
            return resp.read().decode("utf-8")                              # bytes -> texto
    except Exception as erro:                                               # sem rede, 404, proxy...
        print(f"   (rede indisponível para {nome}: {type(erro).__name__}) -> uso a cópia embutida")
        return None

def sha(texto):
    """SHA-256 de um texto em UTF-8 (o mesmo que sha256sum daria no arquivo)."""
    return hashlib.sha256(texto.encode("utf-8")).hexdigest()

PACOTE = {}                                   # nome do arquivo -> texto que vamos analisar
for nome, copia in EMBUTIDO.items():
    online = baixar(nome)
    if online is None:                         # plano B
        PACOTE[nome] = copia
        origem = "embutido"
    else:                                      # plano A deu certo: confere com a cópia
        PACOTE[nome] = online
        origem = "GitHub" if sha(online) == sha(copia) else "GitHub (DIFERENTE da cópia embutida!)"
    print(f"{nome:32} {sha(PACOTE[nome])[:16]}…  [{origem}]")

## 1. Pontuação com citação obrigatória

A regra da Fase 5 e da skill: **nunca marque um mecanismo como presente sem a evidência**. A função abaixo transforma essa regra em código: ela procura a citação dentro do texto da mensagem (ignorando maiúsculas e espaços). Se não achar, recusa a marcação.

In [ ]:
import re

def normalizar(t):
    """Minúsculas e espaços colapsados: a citação não pode falhar por quebra de linha."""
    return re.sub(r"\s+", " ", t).strip().lower()

def pontuar(texto_msg, mecanismos):
    """mecanismos: lista de (nome, presente?, citação_ou_motivo). Devolve a ficha validada."""
    alvo = normalizar(texto_msg)
    ficha = []
    for nome, presente, evid in mecanismos:
        if presente and len(evid.strip()) < 8:                   # "1" ou "sim" não é citação
            ficha.append((nome, "RECUSADO", f"citação curta demais: {evid!r}"))
        elif presente and normalizar(evid) not in alvo:          # citação não está no texto
            ficha.append((nome, "RECUSADO", f"citação não encontrada: {evid!r}"))
        elif not presente and not evid.strip():                  # ausente sem motivo
            ficha.append((nome, "RECUSADO", "ausente sem motivo"))
        else:
            ficha.append((nome, "presente" if presente else "ausente", evid))
    return ficha

def mostrar(titulo, ficha):
    print(f"\n=== {titulo}")
    for nome, estado, evid in ficha:
        print(f"  {estado:9} {nome:42} {evid[:90]}")
    p = sum(1 for _, e, _ in ficha if e == "presente")
    print(f"  -> {p} de {len(ficha)} presentes; {sum(1 for _, e, _ in ficha if e == 'RECUSADO')} recusado(s)")

In [ ]:
P2 = PACOTE["02-suspicious-email.eml"]
M3 = PACOTE["03-related-messages.md"].split("## M3")[1].split("## M4")[0]
M6 = PACOTE["03-related-messages.md"].split("## M6")[1].split("## M7")[0]

mostrar("P2 — e-mail de phishing (exemplo resolvido do original)", pontuar(P2, [
    ("BEC: domínio imitação", True, "dana.whitlock@castellan-hardwood.example"),
    ("BEC: autoridade e urgência", True, "as the old account closes on the 16th"),
    ("BEC: contorno de processo", True, "Our office phones are being migrated to a new system this week"),
    ("BEC: sigilo", False, "Nada pede segredo; mudança bancária é papelada de rotina"),
    ("Phishing: link não bate", True, "cstl-docshare.example/view/r"),
    ("Phishing: saudação genérica", False, "'Hi Jordan,' e o link codifica o username: direcionado"),
]))

mostrar("M3 — e-mail INTERNO para a controladora", pontuar(M3, [
    ("BEC: domínio imitação", False, "Vem da caixa REAL do jpike: o repasse interno tornou isso desnecessário"),
    ("BEC: autoridade (colega de confiança)", True, "From: Jordan Pike <jpike@orrinvalley.example>"),
    ("BEC: urgência", True, "Can you update the vendor master before Friday's run?"),
    ("BEC: contorno de processo", True, "so no need to call, she said email is best"),
    ("BEC: sigilo", False, "Nada pede segredo"),
]))

mostrar("M6 — follow-up do atacante (teste: uma citação errada de propósito)", pontuar(M6, [
    ("BEC: urgência", True, "as soon as the payment is released tomorrow"),
    ("BEC: autoridade emprestada", True, "Our auditors need it for the account transition"),
    ("Phishing: link não bate", True, "click here to confirm"),        # <- não existe no M6!
]))

Repare no **RECUSADO** do M6: a citação "click here to confirm" não existe na mensagem. É assim que a regra "nunca marque presente sem evidência" se defende sozinha.

### 1.1 🎮 Sua vez: o M6

Para cada mecanismo, digite **S** se está presente (e depois a citação) ou **N** se está ausente (e depois o motivo).

In [ ]:
MODO_JOGO = True
mecs_m6 = ["BEC: domínio imitação", "BEC: contorno de processo", "BEC: sigilo", "Phishing: pede credencial"]
sugestao_m6 = {
    "BEC: domínio imitação": (True, "dana.whitlock@castellan-hardwood.example"),
    "BEC: contorno de processo": (False, "Não fecha canal nenhum; só pede o aviso de remessa"),
    "BEC: sigilo": (False, "Nada pede segredo"),
    "Phishing: pede credencial": (False, "Não há link nem pedido de senha: a credencial já foi roubada"),
}
respostas = []
for m in mecs_m6:
    if MODO_JOGO:
        sn = input(f"{m}: presente? (S/N) ").strip().upper()[:1]
        evid = input("   citação (se S) ou motivo (se N): ").strip()
        respostas.append((m, sn == "S", evid))
    else:
        respostas.append((m, *sugestao_m6[m]))
mostrar("M6 — a sua pontuação", pontuar(M6, respostas))
print("\nSugestão para comparar:")
for m, (p, e) in sugestao_m6.items():
    print(f"  {'presente' if p else 'ausente':9} {m:30} {e}")

## 2. A carta bancária como documento

Comparamos a carta (P3 M3) com o que sabemos da Castellan real: a fatura (M1), o site arquivado (P4) e a lista telefônica (P6 L9).

In [ ]:
carta = M3.split("CASTELLAN HARDWOOD SUPPLY LTD.")[1].split("Document metadata")[0]
meta = dict(re.findall(r"^(Create Date|Modify Date|Title|Author)\s*:\s*(.+)$", M3, re.M))
m1 = PACOTE["03-related-messages.md"].split("## M1")[1].split("## M2")[0]
p4 = PACOTE["04-vendor-site-archived.md"]
l9 = PACOTE["06-lookup-results.md"].split("## L9")[1]

checagens = [
    ("Nome da empresa", "Ltd." , "LTD" in M3 and "Inc." in m1,
     "Carta e ERP: 'Ltd.'; fatura M1 e site P4: 'Inc.'", "Sim, comparando com a fatura no arquivo"),
    ("Diretor financeiro", "G. Castellan", "G. Castellan" in carta and "Ines Castellan-Rowe, Chief Financial Officer" in p4,
     "Carta: 'G. Castellan, CFO'; P4: Ines Castellan-Rowe é a CFO", "Talvez: exige conhecer a equipe do fornecedor"),
    ("Telefone", "555-0148", "555-0148" in carta and "555-0112" in m1,
     "Carta: 555-0148 (VoIP sem listagem, L9); M1/P4: 555-0112 (fixo listado desde 1998)", "Sim, se ligasse para o número do cadastro"),
    ("E-mail de contato", "ar@castellan-hardwood", "castellan-hardwood.example" in carta,
     "Carta usa o domínio com hífen; M1 usa castellanhardwood.example", "Sim, com atenção ao domínio"),
    ("Saudação", "valued customers", "To our valued customers" in carta,
     "Genérica: 'To our valued customers', não a Orrin Valley nem a CHS-20417", "Sim"),
    ("Datas dos metadados", "Create Date", True,
     f"Criada {meta.get('Create Date')}; modificada {meta.get('Modify Date')}; título {meta.get('Title')}", "Não: precisa abrir as propriedades do PDF"),
    ("Política do próprio fornecedor", "never change", "never change its banking details by email alone" in p4,
     "O site real (P4) diz que a Castellan nunca muda dados bancários só por e-mail", "Sim, se lesse o site real"),
]
for i, (item, chave, confirmado, detalhe, perceptivel) in enumerate(checagens, 1):
    print(f"{i}. {item:30} {'✔' if confirmado else '?'}  {detalhe}")
    print(f"   Um aprovador cuidadoso perceberia sem ferramentas? {perceptivel}")

## 3. O diagrama de raias, gerado como código (Mermaid)

Cada evento da linha do tempo do Dia 86 vira uma seta. Os pontos de quebra entram como blocos vermelhos. Edite a lista e rode de novo: o texto sai pronto para colar num bloco ` ```mermaid ` do GitHub ou no [mermaid.live](https://mermaid.live/).

In [ ]:
RAIAS = [("INF", "Infraestrutura do atacante"), ("JMB", "Caixa do Jordan<br/>(controle do atacante)"),
         ("J", "Jordan"), ("M", "Maren"), ("ERP", "ERP"), ("CAS", "Castellan (real)")]
EVENTOS = [  # (de, para, rótulo com #58; no lugar de ':', ponto_de_quebra ou None)
    ("CAS", "J", "M1 · 3 mar 14#58;47 · fatura real", None),
    ("INF", "J", "P2 · 10 mar 14#58;02 · phishing na conversa da fatura", "Gateway marca [EXTERNAL] mas não sinaliza domínio imitação (P1)"),
    ("J", "INF", "POST da senha · 10 mar 14#58;20", None),
    ("INF", "JMB", "login · 10 mar 14#58;31", "MFA no jpike. Ausente: P8 MFA=NotRegistered; P1 implantação 70%"),
    ("JMB", "JMB", "regra oculta · 10 mar 14#58;36", None),
    ("JMB", "M", "M3 · 11 mar 13#58;52 · carta bancária", "Processo: ligar para o número do CADASTRO. M3: 'no need to call'"),
    ("M", "ERP", "cadastro alterado · 11 mar", "Processo: dupla aprovação de mudança bancária. Sem log de ERP (P1)"),
    ("M", "JMB", "M4 · 11 mar 15#58;06 · aprovado (escondido pela regra)", None),
    ("CAS", "JMB", "M5 · 12 mar 16#58;05 · 'dados inalterados' (escondido)", "Lembrete do fornecedor: a regra o desviou para RSS Subscriptions (P8)"),
    ("INF", "JMB", "M6 · 12 mar 16#58;40 · follow-up (escondido)", None),
    ("ERP", "INF", "M7 · 13 mar 19#58;02 · aviso de remessa para ar@ do atacante", None),
    ("CAS", "J", "ligação · 16 mar ~14#58;15 · 'não recebemos'", None),
]

def raias_mermaid(raias, eventos):
    linhas = ["sequenceDiagram"] + [f"    participant {k} as {v}" for k, v in raias]
    for de, para, rot, quebra in eventos:
        if quebra:
            linhas.append("    rect rgba(214, 39, 40, 0.15)")
        linhas.append(f"    {de}->>{para}: {rot}")
        if quebra:
            linhas.append(f"    Note over {de},{para}: PONTO DE QUEBRA · {quebra.replace(':', '#58;')}")
            linhas.append("    end")
    return "\n".join(linhas)

texto_raias = raias_mermaid(RAIAS, EVENTOS)
print(texto_raias)
print(f"\nPontos de quebra: {sum(1 for e in EVENTOS if e[3])} (checkpoint: pelo menos 4, um de processo)")

## 4. O cartão de padrão e o detector de vazamento

O cartão não pode conter **nenhum nome, domínio, IP, valor ou data** do caso. O detector abaixo procura esses identificadores no seu texto. Escreva o cartão no formulário e rode.

In [ ]:
#@title Cartão de padrão { display-mode: "form" }
mecanismo = "O operador entra numa conversa real entre cliente e fornecedor, rouba a senha de quem paga e usa a caixa dele para pedir a mudança bancária a um aprovador interno."  #@param {type:"string"}
etapas = "1 domínio imitação e página de coleta; 2 resposta na conversa real; 3 login com a senha; 4 regra oculta; 5 pedido interno; 6 aprovação; 7 pagamento"  #@param {type:"string"}
evidencia = "Logs de auditoria: login de IP/UA novo após clique; New-InboxRule com palavras do fornecedor; Send sem pedido no proxy"  #@param {type:"string"}
deteccao = "Alerta de regra que move mensagens com o nome de fornecedor; domínio recém-registrado parecido com fornecedor"  #@param {type:"string"}
controles = "MFA; ligar para número do cadastro antes de mudar conta; dupla aprovação; alerta de domínio imitação"  #@param {type:"string"}
pistas_falsas = "IP de hospedagem compartilhada; favicon de modelo público; password spray simultâneo sem sucesso"  #@param {type:"string"}

cartao = (f"## Sequestro de conversa com fornecedor e repasse interno (BEC)\n"
          f"Mecanismo: {mecanismo}\nEtapas: {etapas}\nEvidência que o confirma: {evidencia}\n"
          f"Oportunidades de detecção: {deteccao}\nControles que o quebram: {controles}\nPistas falsas comuns: {pistas_falsas}\n")

PROIBIDOS = ["Castellan", "Orrin", "Jordan", "Maren", "Dana", "Marrowline", "Quennmarsh", "Pellbrook", "CHS-", "8830",
             "Serra Azul", "Coopvale", "Pescador", "Antônio"]
def vazamentos(texto, extras=()):
    achados = [p for p in list(PROIBIDOS) + list(extras) if p.lower() in texto.lower()]
    achados += re.findall(r"\b\d{1,3}(?:\.\d{1,3}){3}\b", texto)                 # IPs
    achados += re.findall(r"\b[\w-]+\.example\b", texto)                         # domínios do caso
    achados += re.findall(r"(?:US)?\$\s?[\d.,]+|R\$\s?[\d.,]+", texto)           # valores
    achados += re.findall(r"\b\d{1,2}\s?(?:mar|out|Mar|Oct)\b|\b2026\b", texto)   # datas
    achados += re.findall(r"\+55 00 00000-\d{4}|555-01\d\d", texto)              # telefones
    return achados

v = vazamentos(cartao)
print(cartao)
print("✅ Nenhum identificador do caso." if not v else f"❌ Vazou: {v} — reescreva essas partes.")

## 5. ➕ A triagem da Operação Isca

In [ ]:
# Dados sintéticos da Operação Isca deste dia (plano B embutido).
ISCA_EMBUTIDO = {
"mensagens-isca.txt": "# FICTÍCIO — Operação Isca (investigador-cyber-90-dias). Nenhuma pessoa, conta, número ou domínio real.\n=== S1 · SMS · 2026-10-05 07:12 BRT · remetente +55 00 00000-0001\nSerra Azul: sua chave Pix foi bloqueada por seguranca. Regularize em serraazul-pix.example\n\n=== S2 · pagina https://serraazul-pix.example/ · capturada 2026-10-05 14:02 UTC\nSua chave Pix foi bloqueada preventivamente. Para desbloquear, confirme seus dados de acesso.\nCampos: CPF, senha do internet banking, celular.\nRodape: (c) Cooperativa Serra Azul. Atendimento exclusivo por este portal ate a regularizacao.\n\n=== S3 · WhatsApp Business \"Serra Azul Atendimento\" · +55 00 00000-0009 · 2026-10-05 09:31 BRT\nOla! Aqui e da Central de Seguranca da Serra Azul. Identificamos que voce ainda nao regularizou sua chave Pix.\nSe nao regularizar em 30 minutos, sua conta sera encerrada. Nao ligue para a agencia: as linhas estao\ncongestionadas por causa do bloqueio. Regularize pelo link: lnk.example/sa7\n"
}

In [ ]:
import urllib.request

RAW_ISCA = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-87/"     # pasta caso-continuo/dados/dia-87 no GitHub

ISCA = {}
for nome, copia in ISCA_EMBUTIDO.items():
    try:                                                     # plano A: versão publicada no repo
        with urllib.request.urlopen(RAW_ISCA + nome, timeout=10) as r:
            ISCA[nome] = r.read().decode("utf-8")
        print(f"{nome}: baixado do GitHub")
    except Exception:                                        # plano B: cópia embutida
        ISCA[nome] = copia
        print(f"{nome}: cópia embutida")

In [ ]:
blocos = {}
for parte in ISCA["mensagens-isca.txt"].split("=== ")[1:]:
    chave = parte.split()[0]                       # "S1", "S2", "S3"
    blocos[chave] = parte

mostrar("S1 — SMS", pontuar(blocos["S1"], [
    ("Identidade não bate (domínio)", True, "serraazul-pix.example"),
    ("Bloqueio/regularização imediata (falsa central)", True, "sua chave Pix foi bloqueada"),
    ("Saudação nominal", False, "Sem nome: disparo em massa, não direcionado"),
    ("Pix para chave de terceiro", False, "O SMS não pede Pix: pede o clique"),
]))
mostrar("S2 — página do site falso", pontuar(blocos["S2"], [
    ("Dados pedidos antes de verificação", True, "senha do internet banking"),
    ("Contorno de processo (canal único)", True, "Atendimento exclusivo por este portal"),
    ("Urgência com prazo", False, "A página não dá prazo; o prazo vem no WhatsApp (S3)"),
]))
mostrar("S3 — WhatsApp 'Serra Azul Atendimento'", pontuar(blocos["S3"], [
    ("Autoridade (falsa central)", True, "Central de Seguranca da Serra Azul"),
    ("Urgência com prazo", True, "Se nao regularizar em 30 minutos, sua conta sera encerrada"),
    ("Contorno de processo", True, "Nao ligue para a agencia"),
    ("Sigilo", False, "Não pede segredo; pede pressa"),
]))

FRASE_CONTROLE = [
    ("Nao ligue para a agencia", "S3", "Cooperado liga para o número do cartão / da agência"),
    ("Atendimento exclusivo por este portal", "S2", "Cooperado usa só o app oficial"),
    ("serraazul-pix.example", "S1", "Cooperado reconhece o domínio oficial serraazul.example"),
    ("30 minutos", "S3", "Tempo para pensar e perguntar a alguém de confiança"),
]
print("\nFrase -> controle contornado")
for frase, fonte, controle in FRASE_CONTROLE:
    print(f"  [{fonte}] '{frase}'  ->  {controle}")

In [ ]:
#@title Cartão de padrão brasileiro { display-mode: "form" }
cartao_br = "Falso bloqueio de chave Pix: SMS em massa com link para domínio parecido com o da instituição; página pede senha e celular; perfil de mensageiro se passando por central reforça com prazo curto e desencoraja ligar para a agência; o operador entra com a senha de um dispositivo novo e faz Pix para contas de laranjas. Controles: bloqueio de login de dispositivo novo, alerta de domínio imitação, campanha 'a instituição nunca pede senha', MED."  #@param {type:"string"}

v = vazamentos(cartao_br)
print("✅ Cartão BR sem identificadores do caso." if not v else f"❌ Vazou: {v}")

from pathlib import Path
Path("notas").mkdir(exist_ok=True)
Path("notas/pattern-card.md").write_text(cartao + "\n" + "## Falso bloqueio de chave Pix (smishing + falsa central)\n" + cartao_br + "\n", encoding="utf-8")
Path("notas/swimlane.mmd").write_text(texto_raias + "\n", encoding="utf-8")
try:
    from google.colab import files
    files.download("notas/pattern-card.md"); files.download("notas/swimlane.mmd")
except ImportError:
    print("(Fora do Colab: notas/pattern-card.md e notas/swimlane.mmd)")

## ✅ Antes de fechar o Dia 87

- [ ] P2, M3, M6 e a carta pontuados; todo "presente" com citação e todo "ausente" com motivo
- [ ] Cinco ou mais inconsistências da carta, com fonte e a pergunta "um aprovador perceberia?"
- [ ] Raias de M1 até a ligação da Castellan, com 4+ pontos de quebra (um de processo)
- [ ] Cartão de padrão sem vazamento (capstone e Isca)
- [ ] Nenhuma nota culpa Jordan, Maren ou os cooperados

**Amanhã:** Dia 88: o grafo final. Toda aresta é uma afirmação, com fonte e confiança. E o que você excluiu também aparece.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>